## CS310 Natural Language Processing
## Lab 8: Supervised Fine-Tuning (SFT) for Instruction Following

In this lab, we will prepare a dataset and dataloaders for supervised instruction fine-tuning (SFT). This is the foundation for tuning a pretrained LLM to follow instructions.

We will work with an instruction dataset and implement:
1. Data loading and formatting
2. Custom dataset class for instruction data
3. Custom collate function with padding and ignore_index
4. Training, validation, and test dataloaders

We will experiment with some existing pretrained and instruction-tuned models, so go ahead and download them first:
- Qwen3-0.6B-base: https://huggingface.co/Qwen/Qwen3-0.6B-Base. This is a model that has only completed the pretraining stage. It is Theoretically not fine-tuned over instruction following dataset.
- Qwen3-0.6B: https://huggingface.co/Qwen/Qwen3-0.6B. This model has been through the complete pretraining & post-training stages, which has established instruction following capabilities.
- GPT2-mini: https://huggingface.co/erwanf/gpt2-mini. This model is ptrtrained on a subset of OpenWebText, and it is not instruction-tuned.

The code is adopted from the [LLMs-from-scratch (ch 07)](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch07)

In [8]:
import json
import os
import torch
from functools import partial
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer

---
## T0. Explore the Instruction Dataset and Models

First, we download and load the instruction dataset. Each entry contains:
- `instruction`: The task description
- `input`: Optional additional context (can be empty)
- `output`: The expected response

In [9]:
# Load the Alpaca dataset from lab12_instruct-tuning
file_path = "./dataset/dataset/alpaca_data.json"

with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print("Number of entries:", len(data))

Number of entries: 52002


In [10]:
# Print an example entry
print("Example entry:")
print(json.dumps(data[0], indent=2))

Example entry:
{
  "instruction": "Give three tips for staying healthy.",
  "input": "",
  "output": "1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule."
}


Note that the `'input'` field can be empty. Print `data[999]` to see an example.

In [11]:
# Print data[999] to see an entry with empty input
print("Another example entry:")
print(json.dumps(data[999], indent=2))

Another example entry:
{
  "instruction": "Describe the flavor profile of the following type of cuisine",
  "input": "Japanese",
  "output": "Japanese cuisine is characterized by its subtle and delicate flavors, featuring a combination of salty, sweet, sour, and umami flavors. It also utilizes fresh ingredients with a focus on preserving their natural flavors."
}


---
### Comparing Base vs. Instruction-Tuned Models

Let us see how a base pretrained model and its instruction-tuned variant respond to the same instruction prompt.

In [36]:
# Determine device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Paths (relative to this notebook). Adjust if your models are elsewhere.
qwen_base_path = "./Qwen3-0.6B-Base"
qwen_instruct_path = "../lab7/Qwen3-0.6B"
gpt2_mini_path = "./gpt2-mini"

# Resolve local path helper: prefer an existing absolute path, else keep a safe relative path.
def _resolve_local_path(p):
    # If not an absolute path and does not start with ./ or ../, make it explicitly relative
    if not os.path.isabs(p) and not p.startswith("./") and not p.startswith("../"):
        p = "./" + p
    absp = os.path.abspath(p)
    if os.path.exists(absp):
        return absp
    # fallback to the original string (may point to a relative path)
    return p

qwen_base_resolved = _resolve_local_path(qwen_base_path)
qwen_instruct_resolved = _resolve_local_path(qwen_instruct_path)
gpt2_mini_resolved = _resolve_local_path(gpt2_mini_path)

print("Resolved model paths:", qwen_base_resolved, qwen_instruct_resolved, gpt2_mini_resolved)

# Load tokenizers and models from local directories (local_files_only=True)
tokenizer_base = AutoTokenizer.from_pretrained(qwen_base_resolved, trust_remote_code=True, local_files_only=True)
tokenizer_instruct = AutoTokenizer.from_pretrained(qwen_instruct_resolved, trust_remote_code=True, local_files_only=True)
tokenizer_gpt2_mini = AutoTokenizer.from_pretrained(gpt2_mini_resolved, trust_remote_code=True, local_files_only=True)

qwen_base = AutoModelForCausalLM.from_pretrained(
    qwen_base_resolved,
    trust_remote_code=True,
    local_files_only=True,
).to(device)

qwen_instruct = AutoModelForCausalLM.from_pretrained(
    qwen_instruct_resolved,
    trust_remote_code=True,
    local_files_only=True,
).to(device)

gpt2_mini = AutoModelForCausalLM.from_pretrained(
    gpt2_mini_resolved,
    trust_remote_code=True,
    local_files_only=True,
).to(device)

print("Models loaded on", device)

Resolved model paths: e:\CS310-Natural-Language-Processing\lab\lab8\Qwen3-0.6B-Base e:\CS310-Natural-Language-Processing\lab\lab7\Qwen3-0.6B e:\CS310-Natural-Language-Processing\lab\lab8\gpt2-mini


Loading weights: 100%|██████████| 52/52 [00:00<00:00, 7525.23it/s]

Models loaded on cpu


In [37]:
# Pick a sample instruction from the dataset
sample = data[50]

# Build the instruction prompt manually 
instruction_text = (
    "Below is an instruction that describes a task. "
    "Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{sample['instruction']}"
)
if sample["input"]:
    instruction_text += f"\n\n### Input:\n{sample['input']}"

# golden_answer = "\n\n### Response:\n" + sample["output"]

print(instruction_text)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence to make it more concise.

### Input:
He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.


Let's call the `generate()` function to generate a response.

*Hint*: Specify the arguments of the function.
- set the `max_new_tokens` value properly; 
- set `do_sample` to `False`; 
- specify the `pad_token_id` as the `eos_token_id` from the tokenizer, etc.

In [38]:
# Base model: feed the raw Alpaca prompt and let it continue
prompt_base = instruction_text + "\n\n### Response:\n"
inputs_base = tokenizer_base(prompt_base, return_tensors="pt").to(device)

output_base = qwen_base.generate(
    **inputs_base,
    ### START YOUR CODE ###
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer_base.eos_token_id
    ### END YOUR CODE ###
)

decoded_base = tokenizer_base.decode(output_base[0], skip_special_tokens=True)
print("=== Qwen3-0.6B-Base Output ===")
print(decoded_base[len(prompt_base):])

=== Qwen3-0.6B-Base Output ===
He ran to the bus stop to catch the bus on time.


The base model's output is surprisingly good. 

It is probably due to the data leakage issue: the Alpaca data is already used as the training data for pretraining the `Qwen3-0.6-Base` model.

---

Now let's see how the instruction-tuned model work.

Because Qwen3-0.6B has actually gone through with a complete post-training procedure, it not only has instruction-following ability, but also advanced reasoning/thinking capabilities. To simplify the case, we use `apply_chat_template()` to disable this advanced function.

In [39]:
# Instruction-tuned model: use the chat template
user_content = instruction_text + "\n\n### Response:"

messages = [{"role": "user", "content": user_content}]
prompt_instruct = tokenizer_instruct.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
print(prompt_instruct)

# You can switch `add_generation_prompt` to `False`, and `enable_thinking` to `True`, to see the difference.

<|im_start|>user
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence to make it more concise.

### Input:
He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.

### Response:<|im_end|>
<|im_start|>assistant
<think>

</think>




The output from the instruction-tuned model:

In [40]:
inputs_instruct = tokenizer_instruct(prompt_instruct, return_tensors="pt").to(device)

output_instruct = qwen_instruct.generate(
    **inputs_instruct,
    ### START YOUR CODE ###
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer_instruct.eos_token_id
    ### END YOUR CODE ###
)

decoded_instruct = tokenizer_instruct.decode(output_instruct[0], skip_special_tokens=False)
# print("=== Qwen3-0.6B (Instruct) Output ===\n")
print(decoded_instruct[len(prompt_instruct):])

He ran to the bus stop to catch the bus.<|im_end|>


As we can see in above examples, when the input is an entry from the Alpaca dataset, the base and instruction-tuned models perform equally well.

---

What if we use a more "out-of-the-distribution" input?

In [41]:
OOD_input = "The best way to learn is to"

# Base
inputs_base = tokenizer_base(OOD_input, return_tensors="pt").to(device)
output_base = qwen_base.generate(
    **inputs_base,
    ### START YOUR CODE ###
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer_base.eos_token_id
    ### END YOUR CODE ###
)
print("=== Qwen3-0.6B-Base Output ===")
print(tokenizer_base.decode(output_base[0], skip_special_tokens=True))

# Instruct
messages = [{"role": "user", "content": OOD_input}]
prompt_instruct = tokenizer_instruct.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
)
inputs_instruct = tokenizer_instruct(prompt_instruct, return_tensors="pt").to(device)
output_instruct = qwen_instruct.generate(
    **inputs_instruct,
    ### START YOUR CODE ###
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer_instruct.eos_token_id
    ### END YOUR CODE ###
)
print()
print("=== Qwen3-0.6B (Instruct) Output ===")
print(tokenizer_instruct.decode(output_instruct[0], skip_special_tokens=False))


=== Qwen3-0.6B-Base Output ===
The best way to learn is to practice. The more you practice, the better you will become. The more you practice, the more you will understand. The more you practice, the more you will be able to apply what you have learned. The more you practice, the more you will be able to solve problems. The more you practice, the more you will be able to think. The more you practice, the more you will be able to learn. The more you practice, the more you will be able to understand. The

=== Qwen3-0.6B (Instruct) Output ===
<|im_start|>user
The best way to learn is to<|im_end|>
<|im_start|>assistant
<think>

</think>

The best way to learn is to **practice consistently, seek feedback, and engage with diverse resources**. This approach fosters growth, improves skills, and builds confidence.<|im_end|>


As we can see, the instruction-tuned model indeed has better generic instruction-following capability than the base model.

---

Lastly, let's see how the mini GPT2 model perform.

In [42]:
# In-domain (ID), using an entry from the Alpaca dataset
ID_prompt = instruction_text + "\n\n### Response:\n" # In-domain
ID_inputs = tokenizer_gpt2_mini(ID_prompt, return_tensors="pt").to(device)

ID_output = gpt2_mini.generate(
    **ID_inputs,
    ### START YOUR CODE ###
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer_gpt2_mini.eos_token_id
    ### END YOUR CODE ###
)
print(tokenizer_gpt2_mini.decode(ID_output[0], skip_special_tokens=True))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Edit the following sentence to make it more concise.

### Input:
He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.

### Response:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:

The following sentence is a simple one:



In [43]:
# Out-of-domain (OOD), using a generic input
OOD_prompt = "The best way to learn is to"
OOD_inputs = tokenizer_gpt2_mini(OOD_prompt, return_tensors="pt").to(device)

OOD_output = gpt2_mini.generate(
    **OOD_inputs,
    ### START YOUR CODE ###
    max_new_tokens=100,
    do_sample=False,
    pad_token_id=tokenizer_gpt2_mini.eos_token_id
    ### END YOUR CODE ###
)
print(tokenizer_gpt2_mini.decode(OOD_output[0], skip_special_tokens=True))

The best way to learn is to learn how to use the tools and tools to make your own.

The best way to learn is to learn how to use the tools and tools to make your own.

The best way to learn is to learn how to use the tools and tools to make your own.

The best way to learn is to learn how to use the tools and tools to make your own.

The best way to learn is to learn how to use the tools and tools to make your own


As we can see, GPT2-mini does not perform well in ID or OOD cases, which indicates it is a good starting point for fine-tuning it with an instruction following dataset.

---
## T1. Implement the Instruction Format Function

We will practice using the Alpaca-style prompt formatting for the instruction finetuning task. 

Complete the `format_input` function to format entries as:

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}  (only if input is not empty)
```

In [44]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    # Format the input (only include when not empty)
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""

    return instruction_text + input_text

In [45]:
# Test: formatted response with input field
model_input = format_input(data[5])
desired_response = f"\n\n### Response:\n{data[5]['output']}"
print(model_input + desired_response)


# You are expected to see the following output:
# Below is an instruction that describes a task. Write a response that appropriately completes the request.

# ### Instruction:
# Identify the odd one out.

# ### Input:
# Twitter, Instagram, Telegram

# ### Response:
# Telegram

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the odd one out.

### Input:
Twitter, Instagram, Telegram

### Response:
Telegram


In [46]:
# Test: formatted response without input field
model_input = format_input(data[1])
desired_response = f"\n\n### Response:\n{data[1]['output']}"
print(model_input + desired_response)


# You are expected to see the following output:
# Below is an instruction that describes a task. Write a response that appropriately completes the request.

# ### Instruction:
# What are the three primary colors?

# ### Response:
# The three primary colors are red, blue, and yellow.

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What are the three primary colors?

### Response:
The three primary colors are red, blue, and yellow.


Next, we split the Alpaca instruction-following dataset into three folds:
- 85% for training
- 10% for testing
- 5% for validation

In [47]:
# Split data into train_data, test_data, val_data
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

### START YOUR CODE ###
train_data = data[:train_portion]
test_data = data[train_portion: train_portion + test_portion]
val_data = data[train_portion + test_portion : ]
### END YOUR CODE ###

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 44201
Validation set length: 2601
Test set length: 5200


---

## T2. Implement the InstructionDataset Class

The fisrt step is to create a PyTorch `Dataset` class that pre-tokenizes all instruction entries. The class consists of several methods:
1. In `__init__()`, tokenize the full text (instruction + response) for each entry. 
    - Call `format_input` to format the instruction and input.
    - Run tokenizer.encode() and append the result to `self.encoded_texts`.
2. Return the tokenized text in `__getitem__`.
3. Return the dataset length in `__len__`.

In [48]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            # Format the instruction and input, by calling `format_input()`
            instruction_plus_input = format_input(entry)

            # Format the response
            response_text = f"\n\n### Response:\n{entry['output']}"

            # Concatenate the above two strings
            full_text = instruction_plus_input + response_text

            # Tokenize the full text, and append to self.encoded_texts
            encoded = tokenizer.encode(full_text, add_special_tokens=False)
            self.encoded_texts.append(encoded)

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

We use the tokenizer from `gpt2-mini` as an example.

In [49]:
# Test the tokenizer and dataset
# tokenizer = tiktoken.get_encoding("gpt2")
tokenizer = AutoTokenizer.from_pretrained(gpt2_mini_path)
print("Padding token ID:", tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

train_dataset = InstructionDataset(train_data[:10], tokenizer)
print("Dataset length:", len(train_dataset))
print("First item length:", len(train_dataset[0]))
print("First item:", train_dataset[0])

# You are expected to see the following output:
# Padding token ID: [50256]
# Dataset length: 10
# First item length: 82
# First item: [21106, 318, 281, 12064, 326, 8477, 257, 4876, 13, 19430, 257, ...

Padding token ID: [50256]
Dataset length: 10
First item length: 82
First item: [21106, 318, 281, 12064, 326, 8477, 257, 4876, 13, 19430, 257, 2882, 326, 20431, 32543, 262, 2581, 13, 198, 198, 21017, 46486, 25, 198, 23318, 1115, 9040, 329, 10589, 5448, 13, 198, 198, 21017, 18261, 25, 198, 16, 13, 47659, 257, 12974, 5496, 290, 787, 1654, 284, 2291, 6088, 286, 15921, 290, 13701, 13, 220, 198, 17, 13, 32900, 7987, 284, 1394, 534, 1767, 4075, 290, 1913, 13, 220, 198, 18, 13, 3497, 1576, 3993, 290, 5529, 257, 6414, 3993, 7269, 13]


---
## T3. Implement the Custom Collate Function

The next important step is to implement a custom collate function that:
1. Pads all sequences in a batch to the same length
2. Creates input and target tensors (targets are input tokens with positions shifted by 1)
3. Replaces padding tokens in targets (except the first padding token) with `ignore_index=-100`
4. Optionally truncates to `allowed_max_length`

The `ignore_index` allows PyTorch's cross-entropy loss to ignore padding positions during training.

In [50]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu",
):
    # Find the longest sequence in the batch
    batch_max_length = max(len(item)+1 for item in batch)

    # Pad and prepare inputs and targets
    inputs_list, targets_list = [], []

    for item in batch:
        # Pad sequence to batch_max_length
        padded = list(item) + [pad_token_id] * (batch_max_length - len(item))

        # Truncate the last token for inputs
        # Shift +1 to the right for targets
        inputs = padded[:-1]
        targets = padded[1:]

        # Replace all but the first padding tokens in targets with ignore_index
        pad_positions = [i for i, v in enumerate(targets) if v == pad_token_id]
        if len(pad_positions) > 1:
            for idx in pad_positions[1:]:
                targets[idx] = ignore_index

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_list.append(inputs)
        targets_list.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    ### START YOUR CODE ###
    inputs_tensor = torch.stack([torch.tensor(x, dtype=torch.long, device=device) for x in inputs_list])
    targets_tensor = torch.stack([torch.tensor(x, dtype=torch.long, device=device) for x in targets_list])
    ### END YOUR CODE ###

    return inputs_tensor, targets_tensor

In [51]:
# Test the collate function
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (inputs_1, inputs_2, inputs_3)

inputs, targets = custom_collate_fn(batch)
print("Inputs:")
print(inputs)
print("\nTargets (note -100 for ignored padding):")
print(targets)

# You are expected to see the following output:
# Inputs:
# tensor([[    0,     1,     2,     3,     4],
#         [    5,     6, 50256, 50256, 50256],
#         [    7,     8,     9, 50256, 50256]])

# Targets (note -100 for ignored padding):
# tensor([[    1,     2,     3,     4, 50256],
#         [    6, 50256,  -100,  -100,  -100],
#         [    8,     9, 50256,  -100,  -100]])

Inputs:
tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])

Targets (note -100 for ignored padding):
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


---
## T4. Create Data Loaders

Now create the training, validation, and test data loaders using:
- The `InstructionDataset` class
- The `custom_collate_fn` with appropriate settings
- Use batch size of 8

In [52]:
# Select device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

# Create customized collate function with device and max length settings
customized_collate_fn = partial( # New function with partial application of the given arguments and keywords.
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

Device: cpu


In [53]:
# Create data loaders
batch_size = 8

torch.manual_seed(123)

# Train loader
train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
)

# Validation loader
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
)

# Test loader
test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
)

Token indices sequence length is longer than the specified maximum sequence length for this model (1035 > 1024). Running this sequence through the model will result in indexing errors


In [54]:
# Verify the data loaders
print("Train loader:")
for i, (inputs, targets) in enumerate(train_loader):
    print(f"  Batch {i}: inputs shape {inputs.shape}, targets shape {targets.shape}")
    if i >= 2:  # Only show first 3 batches
        print("  ...")
        break

Train loader:
  Batch 0: inputs shape torch.Size([8, 163]), targets shape torch.Size([8, 163])
  Batch 1: inputs shape torch.Size([8, 300]), targets shape torch.Size([8, 300])
  Batch 2: inputs shape torch.Size([8, 209]), targets shape torch.Size([8, 209])
  ...


In [55]:
# Inspect a sample batch
for inputs, targets in train_loader:
    print("First input in batch:")
    print(inputs[0])
    print("\nFirst target in batch (note -100 for ignored positions):")
    print(targets[0])
    break

First input in batch:
tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 13065,  3876,  1096,   262,  2995,   286,
         2159,  1810,  2873,   287,   530,  6827,    13,   198,   198, 21017,
        18261,    25,   198, 10603,  1810,  2873,   373,   257,  3298,  5358,
          326,  2540,   287, 24414,   290, 15436,  1566, 15761,  7411,   262,
         3741,   286,   262,   995,   338,  7027,   290,  7186,   287,   262,
         7040,   286,   625,  4317,  1510,   661,    13, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256,